# 07 · Placement evaluation

How well does the embedding placement (`api/embeddings.py`) find someone's job from what they say they do?

Plan:
1. **Setup**: load the app's own `Placer`, so we test exactly what ships.
2. **Test set**: pseudo-CVs. For ~300 O*NET occupations, take k of their task statements as "what I do"; the true answer is that occupation's census code.
3. **Baseline**: score with the app's current method. *Stop and look at the numbers before changing anything.*
4. **Robustness**: add one generic phrase ("Mentor junior colleagues") to each pseudo-CV; how much weight does the true job lose?
5. **Variants**: softer τ, statements weighted by specificity, dropping generic statements, text weight by amount. All are different ways of turning the same scores into weights, so we encode once.
6. **Later**: real text (survey free-text answers, real CVs), then fine-tuning.

**Leakage**: the sampled task sentences are in the index themselves (similarity 1.0), so mask them when scoring. The occupation's other tasks stay in, like a real person whose CV paraphrases part of their job.

Caveat: pseudo-CVs use O*NET's own wording, so they are easier than real text. Step 6 matters.

In [13]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
from api.embeddings import Placer, TOP_TASKS_PER_OCCUPATION, TEMPERATURE

census = pd.read_parquet(ROOT / 'api/assets/census.parquet')
codes = (census['census'] if 'census' in census.columns else census.index).to_numpy()
placer = Placer(codes)

index = json.loads((ROOT / 'api/assets/embed/task-index.json').read_text())
task_onet = np.array(index['onet'])
descriptions = set(pd.read_csv(ROOT / 'data/raw/db_31_0_csv/occupation_data.csv')['Description'])
is_description = np.array([t in descriptions for t in placer.task_text])   # occupation descriptions, not tasks

print(len(placer.task_text), 'index rows,', is_description.sum(), 'descriptions,',
      len(placer.group_rows), 'O*NET occupations')

19502 index rows, 913 descriptions, 910 O*NET occupations


In [14]:
print(task_onet[is_description])
task_onet.shape

['51-4031.00' '51-4072.00' '51-9032.00' '11-1011.00' '11-1011.03'
 '11-1021.00' '11-2011.00' '11-2021.00' '11-2022.00' '11-2032.00'
 '11-2033.00' '11-3012.00' '11-3013.00' '11-3013.01' '11-3021.00'
 '11-3031.00' '11-3031.01' '11-3031.03' '11-3051.00' '11-3051.01'
 '11-3051.02' '11-3051.03' '11-3051.04' '11-3051.06' '11-3061.00'
 '11-3071.00' '11-3071.04' '11-3111.00' '11-3121.00' '11-3131.00'
 '11-9013.00' '11-9021.00' '11-9031.00' '11-9032.00' '11-9033.00'
 '11-9041.00' '11-9041.01' '11-9051.00' '11-9071.00' '11-9072.00'
 '11-9081.00' '11-9111.00' '11-9121.00' '11-9121.01' '11-9121.02'
 '11-9131.00' '11-9141.00' '11-9151.00' '11-9161.00' '11-9171.00'
 '11-9179.01' '11-9179.02' '11-9199.01' '11-9199.02' '11-9199.08'
 '11-9199.09' '11-9199.10' '11-9199.11' '13-1011.00' '13-1021.00'
 '13-1022.00' '13-1023.00' '13-1031.00' '13-1032.00' '13-1041.00'
 '13-1041.01' '13-1041.03' '13-1041.04' '13-1041.06' '13-1041.07'
 '13-1041.08' '13-1051.00' '13-1071.00' '13-1074.00' '13-1075.00'
 '13-1081.

(19502,)

## Scoring (same maths as the app, plus masking)

In [15]:
def statement_scores(statements, masked_rows=()):
    """statements x O*NET occupations: mean of each occupation's 2 closest tasks, as in the app."""
    sims = placer.embedder.encode(statements) @ placer.vectors.T
    if len(masked_rows):
        sims[:, list(masked_rows)] = -1.0  # hide the sampled sentences themselves
    grouped = sims[:, placer.task_order]
    bounds = list(placer.group_starts) + [grouped.shape[1]]
    scores = np.empty((len(statements), len(placer.group_rows)))
    for g in range(len(placer.group_rows)):
        block = grouped[:, bounds[g] : bounds[g + 1]]
        n = min(TOP_TASKS_PER_OCCUPATION, block.shape[1])
        scores[:, g] = np.sort(block, axis=1)[:, -n:].mean(axis=1)
    return scores


def to_census(onet_scores):
    """Best O*NET member per census occupation, as in the app."""
    out = np.full(placer.n, -np.inf)
    np.maximum.at(out, placer.group_rows, onet_scores)
    return out


def baseline(scores, tau=TEMPERATURE, weights=None, k=5):
    c = to_census(np.average(scores, axis=0, weights=weights))
    finite = np.isfinite(c)
    w = np.where(finite, np.exp((c - c[finite].max()) / tau), 0.0)
    top = np.argsort(-w)[:k]
    out = np.zeros_like(w)
    out[top] = w[top] / w[top].sum()  # app: keep the top 5, renormalise
    return out

## Test set: pseudo-CVs

In [16]:
rng = np.random.default_rng(0)
N_TEST = 300

# O*NET codes with enough task statements (descriptions excluded).
# Named onet_codes so it doesn't overwrite `codes` (the census codes from the setup cell).
onet_codes, counts = np.unique(task_onet[~is_description], return_counts=True)


def build_test(k_statements, n_test=N_TEST, seed=0):
    """Pseudo-CVs: k task statements from each of n_test random O*NET occupations, scored once."""
    rng = np.random.default_rng(seed)
    candidates = onet_codes[counts >= k_statements + 3]
    cases = []
    for code in rng.choice(candidates, n_test, replace=False):
        rows = np.flatnonzero((task_onet == code) & ~is_description)   # this code's task rows
        sampled = rng.choice(rows, k_statements, replace=False)
        statements = [placer.task_text[i] for i in sampled]
        cases.append({
            "onet": code,
            "census": int(placer.task_census[sampled[0]]),
            "rows": sampled,
            "statements": statements,
            "scores": statement_scores(statements, sampled),
        })
    return cases


test = build_test(2)     # a short description: the hard case
test5 = build_test(5)    # a few CV lines
len(test), len(test5)

(300, 300)

In [17]:
pd.DataFrame(test)

,onet,census,rows,statements,scores
0,35-1012.00,4010,"[10723, 10718]",[Purchase or requisition supplies and equipmen...,"[[0.46056586503982544, 0.34159010648727417, 0...."
1,19-3099.01,1860,"[4971, 4972]",[Review development plans for transportation s...,"[[0.47834837436676025, 0.5886142253875732, 0.4..."
2,29-1299.02,3260,"[9191, 9192]","[Evaluate, diagnose, or treat disorders of the...","[[0.3175504505634308, 0.24681265652179718, 0.2..."
3,43-4041.00,5230,"[12484, 12478]",[Review individual or commercial customer file...,"[[0.3143477737903595, 0.24910126626491547, 0.2..."
4,31-1133.00,3600,"[9910, 9913]",[Accompany patients to and from wards for medi...,"[[0.43229424953460693, 0.2959955036640167, 0.3..."
...,...,...,...,...,...
295,13-1022.00,520,"[1218, 1216]",[Determine which products should be featured i...,"[[0.36548060178756714, 0.2898663282394409, 0.3..."
296,33-2021.00,3750,"[10301, 10292]",[Inspect and test fire protection or fire dete...,"[[0.298612505197525, 0.2792983651161194, 0.269..."
297,39-5011.00,4500,"[11493, 11500]","[Order supplies., Measure, fit, and groom hair...","[[0.3713252544403076, 0.2136831283569336, 0.42..."
298,25-2031.00,2320,"[6795, 6808]","[Assign and grade class work and homework., At...","[[0.3474733233451843, 0.2761305272579193, 0.39..."


## Metrics

In [18]:
def evaluate(method, cases):
    # TODO: for each case: w = method(case['scores']); true = placer.position[case['census']]
    #   rank of the true census (1 = top), weight on the true census
    # return a one-row summary: hit@1, hit@5, MRR (mean reciprocal rank: mean of 1/rank), mean true weight
    ranks, weights = [], []
    for case in cases:
        w = method(case['scores'])
        true = placer.position[case['census']]
        ranks.append(int((w > w[true]).sum()) + 1)     # 1 = placed first
        weights.append(w[true])
    ranks = np.array(ranks)
    return pd.Series({
        'hit@1': (ranks == 1).mean(),       # mean of a mask = share of True
        'hit@5': (ranks <= 5).mean(),
        'MRR': (1 / ranks).mean(),
        'mean true weight': np.mean(weights),
    })


evaluate(baseline, test)

hit@1               0.426667
hit@5               0.700000
MRR                 0.578111
mean true weight    0.371997
dtype: float64

## Robustness: one generic phrase

Add one of these to each pseudo-CV and compare with the clean version: change in rank and in weight on the true job.

In [19]:
GENERIC = [
    "Mentor junior colleagues",
    "Manage the team's budget",
    "Plan weekly schedules",
    "Review documents written by others",
    "Communicate with stakeholders",
]
generic_scores = statement_scores(GENERIC)   # no masking needed, encode once


def add_generic(cases):
    """Each case plus one generic phrase (cycling through the list); the clean cases stay untouched."""
    return [dict(c, scores=np.vstack([c["scores"], generic_scores[i % len(GENERIC)]])) for i, c in enumerate(cases)]


noisy = add_generic(test)
noisy5 = add_generic(test5)

pd.DataFrame({
    ("K=2", "clean"): evaluate(baseline, test),
    ("K=2", "noisy"): evaluate(baseline, noisy),
    ("K=5", "clean"): evaluate(baseline, test5),
    ("K=5", "noisy"): evaluate(baseline, noisy5),
}).T.round(3)

hit@1  hit@5    MRR  mean true weight
K=2 clean  0.427  0.700  0.578             0.372
    noisy  0.293  0.593  0.469             0.239
K=5 clean  0.450  0.767  0.613             0.386
    noisy  0.427  0.730  0.592             0.345

## Variants

Each takes the statements x O*NET score matrix and returns census weights, like `baseline`. All are judged on the K=2 sets: clean and with one generic phrase.

- **τ (softmax temperature)** can't change hit@1 or MRR (the softmax keeps the order); it only changes how weight is split among the top 5, so read *mean true weight* for it.
- **Specificity**: how much a statement's best match stands out (best minus median score).
- **Breadth**: how many different SOC major groups a statement's top-20 occupations come from. Generic phrases spread across fields.

In [20]:
def compare(variants, sets=None):
    sets = sets or {"clean": test, "noisy": noisy}
    return pd.DataFrame({
        (name, data): evaluate(fn, cases)[["hit@1", "MRR", "mean true weight"]]
        for name, fn in variants.items()
        for data, cases in sets.items()
    }).T.round(3)


def specificity_weights(scores):
    best = scores.max(axis=1)
    median = np.median(scores, axis=1)
    return (best - median) / (best + median)


def drop_generic(scores, threshold):
    spec = specificity_weights(scores)
    keep = spec >= threshold
    if not keep.any():
        keep = spec == spec.max()   # never drop everything
    return scores[keep]


group_major = census["soc_major"].to_numpy()[placer.group_rows]   # SOC major group per O*NET column in scores


def breadth(scores, top=20):
    top_cols = np.argsort(-scores, axis=1)[:, :top]
    return np.array([len(set(group_major[row])) for row in top_cols])


print("breadth, generic phrases:", breadth(generic_scores))
print("breadth, real statements (first 20 cases):", np.concatenate([breadth(c["scores"]) for c in test[:20]]))

compare({
    "baseline (tau 0.02)": baseline,
    "tau 0.05": lambda s: baseline(s, 0.05),
    "tau 0.1": lambda s: baseline(s, 0.1),
    "specificity": lambda s: baseline(s, weights=specificity_weights(s)),
    "drop generic (0.5)": lambda s: baseline(drop_generic(s, 0.5)),
    "1 / breadth": lambda s: baseline(s, weights=1 / breadth(s)),
    "1 / breadth^2": lambda s: baseline(s, weights=1 / breadth(s) ** 2),
})

breadth, generic phrases: [ 4  8 11  8  8]
breadth, real statements (first 20 cases): [11 12  8  6  2  2  4  5  4  6  3  6  5  5  4  4  6  7  4  1  6  7  4  3
  3  4 10  3  5  5  4  9  3  5  4  6  7  7  5  8]


hit@1    MRR  mean true weight
baseline (tau 0.02) clean  0.427  0.578             0.372
                    noisy  0.293  0.469             0.239
tau 0.05            clean  0.427  0.578             0.284
                    noisy  0.293  0.469             0.183
tau 0.1             clean  0.427  0.578             0.216
                    noisy  0.293  0.469             0.151
specificity         clean  0.407  0.568             0.371
                    noisy  0.267  0.441             0.218
drop generic (0.5)  clean  0.333  0.501             0.311
                    noisy  0.177  0.343             0.170
1 / breadth         clean  0.413  0.577             0.377
                    noisy  0.310  0.489             0.265
1 / breadth^2       clean  0.407  0.570             0.375
                    noisy  0.297  0.478             0.267

## Title prior

In the app people nearly always give a job title. Today the text is mixed 50/50 with the title, whatever the text says.
The alternative: the title defines a neighbourhood (skill similarity to the title), and the text decides where in that neighbourhood you sit; jumping to a distant field takes strong text evidence.

Two scenarios:
- **Right title**: the case's own census code. The text shouldn't drag you away, even with a generic phrase.
- **Near title**: the most similar job that isn't the true one. Can the text correct a slightly wrong title?

In [21]:
sim = pd.read_parquet(ROOT / "data/interim/census-similarity-ks.parquet").set_index("census")
sim.columns = sim.columns.astype(int)
sim_matrix = sim.reindex(index=codes, columns=codes).to_numpy()   # rows and columns in the same order as `codes`


def softmax_top(c, tau=TEMPERATURE, k=5):
    finite = np.isfinite(c)
    w = np.where(finite, np.exp((c - c[finite].max()) / tau), 0.0)
    top = np.argsort(-w)[:k]
    out = np.zeros_like(w)
    out[top] = w[top] / w[top].sum()
    return out


def with_title(scores, title_census, strength=1.0, tau=TEMPERATURE, k=5):
    """Text scores plus a bonus for jobs similar to the title."""
    c = to_census(scores.mean(axis=0))
    prior = sim_matrix[placer.position[title_census]]   # similarity of every job to the title
    return softmax_top(c + strength * np.nan_to_num(prior), tau, k)


def app_mix(scores, title_census):
    """What the app does today: 50% the title itself, 50% the text placement."""
    out = 0.5 * baseline(scores)
    out[placer.position[title_census]] += 0.5
    return out


def mix_prior(scores, title_census, title_weight=0.5, strength=0.25):
    """What the app does now: the title keeps a fixed share, the text places the rest with a mild title prior.
    Keeps hybrids and domain signal (a strong prior alone collapses everything onto the title)."""
    out = (1 - title_weight) * with_title(scores, title_census, strength)
    out[placer.position[title_census]] += title_weight
    return out


def neighbour_title(census_code):
    """The most similar job that isn't this one (index 0 is the job itself)."""
    row = np.nan_to_num(sim_matrix[placer.position[census_code]], nan=-1)
    return int(codes[np.argsort(-row)[1]])


def titled(method, cases, title_of):
    """Like evaluate, for methods that also take the title."""
    ranks, weights = [], []
    for case in cases:
        w = method(case["scores"], title_of(case))
        true = placer.position[case["census"]]
        ranks.append(int((w > w[true]).sum()) + 1)
        weights.append(w[true])
    ranks = np.array(ranks)
    return pd.Series({"hit@1": (ranks == 1).mean(), "MRR": (1 / ranks).mean(), "mean true weight": np.mean(weights)})


right = lambda case: case["census"]
near = lambda case: neighbour_title(case["census"])

title_methods = {
    "app 50/50": app_mix,
    "50/50 + prior 0.25 in text (app now)": mix_prior,
    "prior 0.1": lambda s, t: with_title(s, t, 0.1),
    "prior 0.25": lambda s, t: with_title(s, t, 0.25),
    "prior 0.5": lambda s, t: with_title(s, t, 0.5),
    "prior 1": lambda s, t: with_title(s, t, 1.0),
}
pd.DataFrame({
    (name, title_name, data): titled(fn, cases, title_of)
    for name, fn in title_methods.items()
    for title_name, title_of in [("right title", right), ("near title", near)]
    for data, cases in [("clean", test), ("noisy", noisy)]
}).T.round(3)

hit@1    MRR  \
app 50/50                            right title clean  1.000  1.000   
                                                 noisy  1.000  1.000   
                                     near title  clean  0.003  0.335   
                                                 noisy  0.003  0.288   
50/50 + prior 0.25 in text (app now) right title clean  1.000  1.000   
                                                 noisy  1.000  1.000   
                                     near title  clean  0.003  0.406   
                                                 noisy  0.003  0.396   
prior 0.1                            right title clean  0.590  0.721   
                                                 noisy  0.550  0.690   
                                     near title  clean  0.513  0.654   
                                                 noisy  0.443  0.604   
prior 0.25                           right title clean  0.777  0.860   
                                                 noisy  0.773  0.857   
                                     near title  clean  0.537  0.692   
                                                 noisy  0.467  0.642   
prior 0.5                            right title clean  0.890  0.937   
                                                 noisy  0.910  0.949   
                                     near title  clean  0.487  0.678   
                                                 noisy  0.360  0.598   
prior 1                              right title clean  0.973  0.987   
                                                 noisy  0.987  0.993   
                                     near title  clean  0.317  0.590   
                                                 noisy  0.213  0.520   

                                                        mean true weight  
app 50/50                            right title clean             0.686  
                                                 noisy             0.619  
                                     near title  clean             0.188  
                                                 noisy             0.121  
50/50 + prior 0.25 in text (app now) right title clean             0.857  
                                                 noisy             0.849  
                                     near title  clean             0.257  
                                                 noisy             0.209  
prior 0.1                            right title clean             0.543  
                                                 noisy             0.469  
                                     near title  clean             0.469  
                                                 noisy             0.371  
prior 0.25                           right title clean             0.714  
                                                 noisy             0.698  
                                     near title  clean             0.510  
                                                 noisy             0.414  
prior 0.5                            right title clean             0.859  
                                                 noisy             0.869  
                                     near title  clean             0.467  
                                                 noisy             0.356  
prior 1                              right title clean             0.955  
                                                 noisy             0.965  
                                     near title  clean             0.315  
                                                 noisy             0.210


## Real people: survey titles and descriptions

17 classmates and friends typed a job title, and 13 described their day-to-day work. `TRUE_JOB` is Tim's judgment of each person's real occupation, in survey order.

1. **Titles**: how often does the app's title search land on the right job? This tells us how much to trust a title, i.e. the prior strength.
2. **Descriptions**: the real-text test. The title from search plus the description, through each method. Uses `split_statements` (no Haiku rewrite), so it's a bit harsher than the app.


In [22]:

from api.engine import Engine
from api.embeddings import split_statements

engine = Engine()
survey = pd.read_csv(ROOT / "data/raw/survey/responses.csv")
survey = survey.rename(columns={survey.columns[1]: "title", survey.columns[2]: "description"})

TRUE_JOB = [   # census code per respondent, in survey order (Tim's assignments)
    110,   # Engineering manager -> computer and information systems managers
    1107,  # Product designer -> IT project managers, QA testers, UI designers & more tech
    1020,  # Software engineer
    1030,  # Senior Web Developer
    1600,  # Research Scientist -> agricultural and food scientists
    5860,  # Office Coordinator -> office clerks, general
    1860,  # Senior Researcher (AI think tank) -> social scientists (incl. political scientists)
    1020,  # Senior Software Engineer
    1610,  # Postdoctoral researcher (biology) -> biological scientists
    1860,  # Consultant (right-wing extremism) -> social scientists (incl. political scientists)
    2010,  # Social worker
    2000,  # Counselor
    2100,  # Lawyer
    2100,  # Sr. Managing counsel employment -> lawyers
    2010,  # Social worker
    1860,  # University researcher in linguistics -> social scientists (linguists are in "all other")
    1030,  # Web developer
]
survey["true"] = TRUE_JOB


def title_outcome(row):
    found = [m["census"] for m in engine.search_jobs(row["title"].strip())]
    top = found[0] if found else None
    neighbours = set(codes[np.argsort(-np.nan_to_num(sim_matrix[placer.position[row["true"]]], nan=-1))[1:6]])
    if top == row["true"]:
        outcome = "right"
    elif top in neighbours:
        outcome = "near miss"     # among the true job's 5 most similar jobs
    else:
        outcome = "wrong"
    return pd.Series({"search top": engine.census.at[top, "title"] if top else None, "top": top,
                      "true in top 8": row["true"] in found, "outcome": outcome})


survey = survey.join(survey.apply(title_outcome, axis=1))
survey["true title"] = survey["true"].map(engine.census["title"])
display(survey[["title", "search top", "true title", "outcome", "true in top 8"]])
survey["outcome"].value_counts()


,title,search top,true title,outcome,true in top 8
0,Engineering manager,Architectural and engineering managers,Computer and information systems managers,wrong,True
1,Product designer,Designers,"IT project managers, QA testers, UI designers ...",wrong,True
2,Software engineer,"Software developers, applications and systems ...","Software developers, applications and systems ...",right,True
3,Senior Web Developer,Web developers,Web developers,right,True
4,Research Scientist,Computer and information research scientists,Agricultural and food scientists,wrong,True
5,Office Coordinator,First-line supervisors of office and administr...,"Office clerks, general",wrong,False
6,Senior Researcher,Mathematicians,"Planners, anthropologists, historians & more",wrong,True
7,Senior Software Engineer,"Software developers, applications and systems ...","Software developers, applications and systems ...",right,True
8,Postdoctoral researcher,Natural sciences managers,Biological scientists,near miss,True
9,Consultant,Financial managers,"Planners, anthropologists, historians & more",wrong,False


outcome
right        8
wrong        7
near miss    2
Name: count, dtype: int64

In [ ]:
from api.cv import activities_from_text

described = survey[survey["description"].fillna("").str.strip().str.len() > 10].copy()
described["statements"] = described["description"].map(lambda d: split_statements(d) or [d.strip()]) # or activites_from_text
described["scores"] = described["statements"].map(statement_scores)

real_methods = {
    "text only": lambda s, t: baseline(s),
    "app 50/50": app_mix,
    "50/50 + prior 0.25 in text (app now)": mix_prior,
    "prior 0.25": lambda s, t: with_title(s, t, 0.25),
    "prior 0.5": lambda s, t: with_title(s, t, 0.5),
    "prior 1": lambda s, t: with_title(s, t, 1.0),
}


def top_pick(method, row):
    w = method(row["scores"], row["top"])
    return engine.census["title"].iat[int(np.argmax(w))], w[placer.position[row["true"]]]


picks = pd.DataFrame({name: described.apply(lambda r: top_pick(fn, r)[0], axis=1) for name, fn in real_methods.items()})
picks.insert(0, "true", described["true title"])
picks.insert(0, "typed title", described["title"])
display(picks)

pd.DataFrame({
    name: {
        "hit@1": (picks[name] == picks["true"]).mean(),
        "mean true weight": described.apply(lambda r: top_pick(fn, r)[1], axis=1).mean(),
    }
    for name, fn in real_methods.items()
}).T.round(3)


,typed title,true,text only,app 50/50,50/50 + prior 0.25 in text (app now),prior 0.25,prior 0.5,prior 1
0,Engineering manager,Computer and information systems managers,Computer programmers,Architectural and engineering managers,Architectural and engineering managers,Electrical and electronics engineers,Architectural and engineering managers,Architectural and engineering managers
1,Product designer,"IT project managers, QA testers, UI designers ...","Sales representatives, wholesale and manufactu...",Designers,Designers,Designers,Designers,Designers
2,Software engineer,"Software developers, applications and systems ...","Switchboard operators, including answering ser...","Software developers, applications and systems ...","Software developers, applications and systems ...",Web developers,Database administrators,"Software developers, applications and systems ..."
5,Office Coordinator,"Office clerks, general","Court, municipal, and license clerks",First-line supervisors of office and administr...,First-line supervisors of office and administr...,First-line supervisors of office and administr...,First-line supervisors of office and administr...,First-line supervisors of office and administr...
6,Senior Researcher,"Planners, anthropologists, historians & more",Postsecondary teachers,Mathematicians,Mathematicians,Statisticians,Mathematicians,Mathematicians
7,Senior Software Engineer,"Software developers, applications and systems ...","Engineering technicians, except drafters","Software developers, applications and systems ...","Software developers, applications and systems ...","IT project managers, QA testers, UI designers ...","Software developers, applications and systems ...","Software developers, applications and systems ..."
9,Consultant,"Planners, anthropologists, historians & more","Planners, anthropologists, historians & more",Financial managers,Financial managers,Financial managers,Financial managers,Financial managers
10,Social worker,Social workers,Childcare workers,Social workers,Social workers,Counselors,Counselors,Counselors
11,counselor,Counselors,Social workers,Psychologists,Psychologists,Social workers,Social workers,Psychologists
12,Lawyer,Lawyers,Secretaries and administrative assistants,Lawyers,Lawyers,Lawyers,Lawyers,Lawyers


,hit@1,mean true weight
text only,0.071,0.079
app 50/50,0.500,0.289
50/50 + prior 0.25 in text (app now),0.500,0.371
prior 0.25,0.214,0.242
prior 0.5,0.286,0.326
prior 1,0.357,0.387


## Findings

TODO (Tim): what held up, what didn't, and what goes into the app.